# Protocol dry run (Section 7)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omega2417/bnt/blob/master/evoproto/notebooks/03_protocol_dry_run.ipynb)

Three arms × one task × ten seeded replicates, then the full analysis plan:
paired Wilcoxon, Holm correction, Cliff's δ, bootstrap CIs.

**Tag: SYNTHETIC.** Arm A is a time-matched stand-in for human designers and
arm C's prior is hand-set, not derived from a curated EE-KG. The run tests the
pipeline, not the hypotheses H1–H3, and no inference about them is drawn.


In [ ]:
# Google Colab: install the package from the repository (skip if running locally)
import importlib.util, subprocess, sys

if importlib.util.find_spec('evoproto') is None:
    IN_COLAB = 'google.colab' in sys.modules
    if IN_COLAB:
        subprocess.run(['pip', 'install', '-q',
                        'git+https://github.com/omega2417/bnt.git#subdirectory=evoproto'],
                       check=True)
    else:
        sys.path.insert(0, '../src')   # running from notebooks/ in a clone

import evoproto
print('evoproto', evoproto.__version__)


## Seeds are derived, not drawn

`blake2b(package_version | task | arm | replicate)`. Reproducible across
machines, and arms cannot silently share a random stream.


In [ ]:
from evoproto.data import stable_seed

for arm in ('A', 'B', 'C'):
    print(arm, [stable_seed('T1', arm, r) for r in range(3)])
print('same call twice:', stable_seed('T1', 'C', 0) == stable_seed('T1', 'C', 0))


## Run the protocol


In [ ]:
from evoproto.experiment import run_protocol

result = run_protocol(tasks=('T1',), replicates=10, population=40, generations=25)
for arm, row in result.table().items():
    print(arm, row)


## The statistical plan

No result is described as significant without its Holm-adjusted p-value **and**
its effect size.


In [ ]:
rows = [{'contrast': r['contrast'], 'metric': r['metric'],
         'p': round(r['p_value'], 4), 'p_holm': round(r['p_holm'], 4),
         'cliffs_delta': round(r['cliffs_delta']['delta'], 2),
         'magnitude': r['cliffs_delta']['magnitude'],
         'significant': r['significant']}
        for r in result.statistics['T1']]

try:                      # pandas is preinstalled on Colab; plain text otherwise
    import pandas as pd
    display(pd.DataFrame(rows))
except ImportError:
    header = list(rows[0])
    print('  '.join(f'{h:>12}' for h in header))
    for row in rows:
        print('  '.join(f'{str(row[h]):>12}' for h in header))


## Power

Ten replicates detect large effects only (Eq. 24) — recorded as a limitation in
Section 8.3, not hidden.


In [ ]:
from evoproto.experiment import required_pairs

for effect in (0.6, 0.8, 1.0, 1.2):
    print(f'effect {effect}: about {required_pairs(effect):.0f} pairs needed')


## Ablation: what does the analog prior actually buy?

Arms B and C differ only in the sampling prior. Compare the generation at which
each first finds a feasible design.


In [ ]:
import numpy as np

for arm in ('B', 'C'):
    its = [c.metrics['ITS'] for c in result.cells if c.arm == arm]
    print(f'arm {arm}: ITS mean {np.mean(its):.2f}, max {np.max(its):.0f}')


## Save the results with their environment record


In [ ]:
directory = result.save('results/dry_run')
print('written to', directory)
print(result.environment)


## Unimplemented tasks fail loudly

T2 and T3 are registered and declared unimplemented rather than quietly
approximated.


In [ ]:
from evoproto.experiment import run_cell, TASKS

for key, task in TASKS.items():
    print(key, '->', 'implemented' if task.implemented else 'NEEDS INPUT')
try:
    run_cell('T2', 'C', 0)
except NotImplementedError as error:
    print('\nrefused:', error)
